# Lesson 17 — Locking, Blocking, Deadlocks & Waits

> **Prerequisite:** Tiers 1–4, especially lesson 11 (transactions & isolation). This lesson shows the *internals and diagnosis* of the concurrency behaviour introduced there.

## What you'll learn
- Lock modes (S, X, U, IS, IX), granularity, and lock escalation
- How to read a blocking chain (who is blocking whom, and on what)
- Deadlocks: how they form, the victim, and reading the deadlock graph
- The waits-and-queues methodology — the master lens for "what is the server waiting on?"

## Setup
Run `setup.sql` once. It creates the `lesson17` schema with a small `Account` table (blocking/deadlock demos) and a larger `BigOrders` table (lock-escalation demo). Re-run to reset.

**Tooling note:** run the runnable SQL cells with Azure Data Studio connected to the mssql-learn container. Several demos need **two concurrent sessions** — in ADS, open a second query editor (or a second connection) to act as *Session B*; outside ADS you can run `.\scripts\connect.ps1` again for a second sqlcmd terminal. Cells marked `-- [manual]` open a transaction or block/deadlock on purpose: run them yourself in the labelled session — they are **excluded from top-to-bottom execution** so a run-all doesn't hang. The deadlock-graph query (Example 5) uses XML methods, which need `QUOTED_IDENTIFIER ON` (set in the preamble).

In [ ]:
USE AdventureWorks2022;
GO
-- XML data type methods (used by the deadlock-graph query below) require QUOTED_IDENTIFIER ON.
SET QUOTED_IDENTIFIER ON;
GO

## Concepts

### Lock modes & granularity
Locks protect data during transactions. Common modes: **S** (shared, reads), **X** (exclusive, writes), **U** (update, the intermediate step), and intent locks **IS/IX** at coarser levels. Locks are taken at row (KEY/RID), page, or object (table) granularity.

### Lock escalation
When a single statement holds too many fine-grained locks (~5000), SQL Server **escalates** them to one coarse lock (usually a table X lock) to save memory — which can suddenly block everyone else.

### Blocking
**Blocking** is normal, temporary contention: session B waits for a lock session A holds. `sys.dm_exec_requests.blocking_session_id` and `sys.dm_os_waiting_tasks` reveal the chain; `sys.dm_tran_locks` shows the exact locks.

### Deadlocks
A **deadlock** is a *cycle*: A holds a lock B wants, while B holds a lock A wants. SQL Server detects the cycle and kills one transaction (the **victim**, error 1205). Deadlock graphs are captured automatically in the `system_health` Extended Events session.

### Waits & queues
Every time a task can't proceed it records a **wait**. `sys.dm_os_wait_stats` (cumulative) and `sys.dm_os_waiting_tasks` (live) tell you what the server spends its time waiting on — the fastest path to a root cause.

## Worked investigations

### Blocking (needs two sessions)

Open a second ADS query editor (or connection) to act as **Session B** — outside ADS, run `.\scripts\connect.ps1` again for a second terminal. The blocks below are labelled `Session A` and `Session B`. The `-- [manual]` cells are excluded from run-all.

In [ ]:
-- [manual] Session A — run in one session; holds an open transaction (excluded from run-all).
-- Session A: start a transaction and hold an exclusive lock (do NOT commit yet)
BEGIN TRANSACTION;
UPDATE lesson17.Account SET Balance = Balance - 100 WHERE AccountID = 1;
-- (leave this open, switch to Session B)

In [ ]:
-- [manual] Session B — run in a SECOND session while Session A's transaction is open.
-- Session B: try to read the same row — it BLOCKS waiting for Session A's lock
SELECT * FROM lesson17.Account WHERE AccountID = 1;   -- hangs until A commits/rolls back

Now, from Session A (a new batch) or a third session, see the blocking chain:

### Example 1 — Who is blocking whom?

Blocking chain via `sys.dm_exec_requests`.

In [ ]:
SELECT
    r.session_id        AS WaitingSession,
    r.blocking_session_id AS BlockedBy,
    r.wait_type,
    r.wait_time         AS WaitMs,
    r.wait_resource,
    t.text              AS WaitingSQL
FROM sys.dm_exec_requests AS r
CROSS APPLY sys.dm_exec_sql_text(r.sql_handle) AS t
WHERE r.blocking_session_id <> 0;

### Example 2 — What locks are currently held / requested?

Held/requested locks via `sys.dm_tran_locks`.

In [ ]:
SELECT
    l.request_session_id  AS SessionId,
    l.resource_type,
    l.request_mode,         -- S, X, U, IS, IX, etc.
    l.request_status,       -- GRANT or WAIT
    OBJECT_NAME(p.object_id) AS ObjectName
FROM sys.dm_tran_locks AS l
LEFT JOIN sys.partitions AS p ON p.hobt_id = l.resource_associated_entity_id
WHERE l.resource_database_id = DB_ID()
ORDER BY l.request_session_id, l.resource_type;

### Example 3 — Waiting tasks (the live view of what is stuck)

Live waiting tasks.

In [ ]:
SELECT
    wt.session_id,
    wt.wait_type,
    wt.wait_duration_ms,
    wt.blocking_session_id
FROM sys.dm_os_waiting_tasks AS wt
WHERE wt.session_id > 50;   -- skip system sessions

In [ ]:
-- [manual] Session A — run after observing the block, to release the lock.
-- Session A: release the lock
COMMIT TRANSACTION;   -- Session B's SELECT now returns immediately

### Lock modes & escalation

### Example 4 — A large UPDATE escalates many row/page locks to a single TABLE lock

Lock escalation to an OBJECT lock. This block opens and rolls back its own transaction, so it is safe to run in one session; watch the escalated lock from a second session while it is open.

In [ ]:
-- Run this and inspect sys.dm_tran_locks (in another session) — you'll see an OBJECT-level X lock.
BEGIN TRANSACTION;
UPDATE lesson17.BigOrders SET Status = Status WHERE SalesOrderID > 0;  -- touches every row
-- In Session B: SELECT resource_type, request_mode FROM sys.dm_tran_locks
--               WHERE request_session_id = <this SPID>;  -- expect an OBJECT X lock (escalated)
ROLLBACK TRANSACTION;

### Deadlock (needs two sessions) — deterministic ordering

Run the two `-- [manual]` blocks in separate sessions, interleaving as the comments describe. One session becomes the victim.

In [ ]:
-- [manual] Session A — deadlock demo; interleave with Session B in a separate session.
-- Session A
BEGIN TRAN;
UPDATE lesson17.Account SET Balance = Balance - 1 WHERE AccountID = 1;  -- locks row 1
-- (pause)                                                                  -- then run the next line
UPDATE lesson17.Account SET Balance = Balance - 1 WHERE AccountID = 2;  -- wants row 2

In [ ]:
-- [manual] Session B — run its first UPDATE during Session A's pause.
-- Session B (run its first UPDATE during A's pause)
BEGIN TRAN;
UPDATE lesson17.Account SET Balance = Balance - 1 WHERE AccountID = 2;  -- locks row 2
UPDATE lesson17.Account SET Balance = Balance - 1 WHERE AccountID = 1;  -- wants row 1 -> DEADLOCK

One session becomes the victim (Msg 1205) and is rolled back automatically.

### Example 5 — Read the most recent deadlock graphs from the system_health Extended Event session

Deadlock graph from `system_health`.

In [ ]:
SELECT
    XEvent.value('(@timestamp)[1]', 'datetime2')           AS DeadlockTime,
    XEvent.query('.')                                       AS DeadlockGraphXml
FROM (
    SELECT CAST(target_data AS XML) AS TargetData
    FROM sys.dm_xe_session_targets st
    JOIN sys.dm_xe_sessions s ON s.address = st.event_session_address
    WHERE s.name = 'system_health' AND st.target_name = 'ring_buffer'
) AS Data
CROSS APPLY TargetData.nodes('RingBufferTarget/event[@name="xml_deadlock_report"]') AS XEventData(XEvent)
ORDER BY DeadlockTime DESC;

### Waits-and-queues methodology

### Example 6 — Top cumulative wait types since the last restart (the master triage query)

Top cumulative waits (the triage query).

In [ ]:
SELECT TOP 10
    wait_type,
    wait_time_ms,
    waiting_tasks_count,
    wait_time_ms / NULLIF(waiting_tasks_count, 0) AS AvgWaitMs
FROM sys.dm_os_wait_stats
WHERE wait_type NOT IN (   -- filter benign/idle waits (the real list is long; these are the common ones)
        'SLEEP_TASK','BROKER_TASK_STOP','XE_TIMER_EVENT','CHECKPOINT_QUEUE',
        'LAZYWRITER_SLEEP','REQUEST_FOR_DEADLOCK_SEARCH','SQLTRACE_INCREMENTAL_FLUSH_SLEEP',
        'WAITFOR','DIRTY_PAGE_POLL','HADR_FILESTREAM_IOMGR_IOCOMPLETION','BROKER_RECEIVE_WAITFOR',
        'SOS_WORK_DISPATCHER','LOGMGR_QUEUE','QDS_PERSIST_TASK_MAIN_LOOP_SLEEP',
        'DISPATCHER_QUEUE_SEMAPHORE','SP_SERVER_DIAGNOSTICS_SLEEP','BROKER_TO_FLUSH',
        'PWAIT_EXTENSIBILITY_CLEANUP_TASK','QDS_ASYNC_QUEUE')
ORDER BY wait_time_ms DESC;

### Example 7 — Reset wait stats (dev only) so you can measure waits for a specific workload

Resetting wait stats to measure a specific workload. Resetting clears the server-wide cumulative counters, so it stays commented — never run it on a shared server. Uncomment and run it (dev only) before measuring a workload:

```sql
DBCC SQLPERF('sys.dm_os_wait_stats', CLEAR);
```

In [ ]:
PRINT 'Uncomment the DBCC SQLPERF line to reset wait stats before measuring a workload.';

## Common issues this explains
- A query "hanging" (it's blocked, not broken).
- Intermittent deadlock errors (1205) under concurrency.
- A bulk update suddenly blocking everyone (lock escalation).

## Pitfalls
- `NOLOCK` "fixes" blocking by reading dirty/duplicated/missing rows — almost never correct.
- `sys.dm_os_wait_stats` is cumulative since restart; reset it (dev) or snapshot-diff it to study one workload.
- Filter idle/benign waits (SLEEP_TASK, LAZYWRITER_SLEEP, etc.) or they drown the signal.
- Deadlock graphs age out of the `system_health` ring buffer — capture promptly.

## Cheatsheet
See `cheatsheets/06-execution-plans.md` (and lesson 11 for isolation levels)

## Exercises
Open `exercises.ipynb` and work them in order (some need two sessions). Solutions in `exercises-solutions.ipynb`.